In [31]:
import geopandas as gpd
import math
import pandas as pd

import networkx as nx
import folium
import osmnx as ox

WALKING_SPEED = 1.31
DAYS_IN_YEAR = 365

In [32]:
def geocode_location(prompt_text):
    """
    :return: geo obj, text location from user input
    """
    while True:
        location = input(prompt_text)
        try:
            # TODO: Ensure location is within city limits
            geo = ox.geocode(location)
            return geo, location
        except Exception:
            print("Location not found. Please query again.")

# Source: ChatGPT
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # meters
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = math.sin(dphi/2)**2 + \
        math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2

    return 2 * R * math.asin(math.sqrt(a))

def loc_prompt():
    """
    :return: start and end geo locations
    """
    geo_start, loc_start = geocode_location("Please enter your starting location: ")
    geo_end, loc_end = geocode_location("Please enter your destination: ")

    print(f"Start Point: {geo_start} \'{loc_start}\'")
    print(f"End Point: {geo_end} \'{loc_end}\'")
    locs = [loc_start, loc_end]
    points = [geo_start, geo_end]
    return points, locs

def center(points):
    """
    :param points:
    :return: center point and the correct radius for a plot
    """
    start_lat_f = points[0][0]
    start_lon_f = points[0][1]
    end_lat_f = points[1][0]
    end_lon_f = points[1][1]

    geo_center = (
        ( float(start_lat_f)+float(end_lat_f) ) / 2,
        ( float(start_lon_f)+float(end_lon_f) ) / 2
                 )

    # Haversine distance between locations * 1.25 for some bezels for mapping
    bezel_factor = 1.25
    geo_dist_f = (haversine(start_lat_f, start_lon_f, end_lat_f, end_lon_f) / 2) * bezel_factor
    # geo_dist = max(geo_dist, 1000)
    geo_dist_f = round(geo_dist_f, 2)

    return geo_center, geo_dist_f, start_lat_f, start_lon_f, end_lat_f, end_lon_f

def compute_route_metrics(graph_f, origin_node_f, destination_node_f, departure_time_f, start, end):

    departure_time_shift_f = assign_shift_string(departure_time_f)
    # --- Compute routes ---
    rc = nx.shortest_path(
        graph_f, origin_node_f, destination_node_f,
        weight='crime_weight', method='dijkstra'
    )

    rs = nx.shortest_path(
        graph_f, origin_node_f, destination_node_f,
        weight='length', method='dijkstra'
    )

    # --- Helper function to compute path length ---
    def path_weight(graph_ff, route, weight_attr):
        total = 0
        for u_f, v_f in zip(route[:-1], route[1:]):
            edge_data = graph_ff.get_edge_data(u_f, v_f)

            # Handle MultiDiGraph (multiple edges)
            if isinstance(edge_data, dict):
                edge_data = min(edge_data.values(), key=lambda x: x.get(weight_attr, float('inf')))

            total += edge_data.get(weight_attr, 0)
        return total

    # --- Compute metrics ---
    crime_route_length_m = path_weight(graph_f, rc, 'length')
    crime_route_weighted = path_weight(graph_f, rc, 'crime_weight')
    crime_route_exposure_count = path_weight(graph_f, rc, 'crime_count')

    short_route_length_m = path_weight(graph_f, rs, 'length')
    short_route_weighted = path_weight(graph_f, rs, 'crime_weight')
    short_route_exposure_count = path_weight(graph_f, rs, 'crime_count')

    # --- Print results cleanly ---
    print("\n" + "="*100)
    print("ROUTE COMPARISON")
    print(f'From \'{start}\' to \'{end}\' at {departure_time_f} EST')
    print("="*100)

    print("\n   [Shortest Distance Route]                    [Safest (Crime-Weighted) Route]")
    print(f"  Total Distance (meters): {short_route_length_m:,.2f}          Total Distance (meters):        {crime_route_length_m:,.2f}")
    print(f"  Crime-Weighted Distance: {short_route_weighted:,.2f}          Crime-Weighted Distance:        {crime_route_weighted:,.2f}")

    print("\n" + "="*100)
    print("DIFFERENCE")
    print("="*100)
    print(f"  Extra Distance for Safety (m):  {crime_route_length_m - short_route_length_m:,.2f}")

    #crime_reduction_score = 0
    shift_len = time_in_shift_seconds(departure_time_shift)

    short_walk_time = short_route_length_m / WALKING_SPEED
    crime_walk_time = crime_route_length_m / WALKING_SPEED

    short_exposure_ratio = short_walk_time / shift_len
    crime_exposure_ratio = crime_walk_time / shift_len

    short_crime_exposure = ( short_exposure_ratio * short_route_exposure_count ) / DAYS_IN_YEAR
    crime_crime_exposure = ( crime_exposure_ratio * crime_route_exposure_count ) / DAYS_IN_YEAR

    # avoid division by 0
    if crime_crime_exposure == 0:
        score = float('inf')  # infinite crime reduction
        print(f"  Crime Reduction Score:          No crime found on route.")
    else:
        score = ( ( crime_crime_exposure - short_crime_exposure ) / short_crime_exposure ) * -100
        print(f"  Decrease in crime exposure at {buffer_dist} meter render distance: {score:,.2f}%")

def dep_time():
    """
    :return: depart_time, the time the user will depart HH:MM (24hr)
    """
    while True:
        depart_time = input('What time will you depart from your starting location?\nEnter value in format HH:MM (24hr): ')
        parts = depart_time.split(':')
        try:
            if len(parts) != 2:
                raise ValueError
            hours = int(parts[0])
            minutes = int(parts[1])
            if not (0 <= hours <= 23) or not (0 <= minutes <= 59):
                raise ValueError
            break  # valid input, exit loop
        except ValueError:
            print("Invalid time format. Please enter in format HH:MM (24hr)")

    return depart_time

def assign_shift_string(hour):
    """
    :return: label for time shift
    :param: string representing hour (0-23)
    """
    parts = hour.split(':')
    hour = int(parts[0])
    if 4 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 16:
        return 'afternoon'
    elif 16 <= hour < 21:
        return 'evening'
    elif 21 <= hour < 24:
        return 'night'
    elif 0 <= hour < 4:
        return 'graveyard'
    return None

def assign_shift_int(hour):
    """
    :return: label for time shift
    :param: integer value for hour (0-23)
    """
    if 4 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 16:
        return 'afternoon'
    elif 16 <= hour < 21:
        return 'evening'
    elif 21 <= hour < 24:
        return 'night'
    elif 0 <= hour < 4:
        return 'graveyard'
    return None

def time_in_shift_seconds(shift):
    """
    :param shift: String representation of what shift we are looking at
    :return: length of that shift in seconds
    """
    shift_hours = {
        'morning': 12 - 4,      # 8 hours
        'afternoon': 16 - 12,   # 4 hours
        'evening': 21 - 16,     # 5 hours
        'night': 24 - 21,       # 3 hours
        'graveyard': 4 - 0      # 4 hours
    }

    if shift not in shift_hours:
        return None

    return shift_hours[shift] * 3600

Import Data


In [33]:
crime_gdf = gpd.read_file("../../../Data/Crime/Crime_Incidents_in_2025.geojson")

# add needed columns
crime_gdf['END_DATE'] = pd.to_datetime(crime_gdf['END_DATE'])
crime_gdf['hour'] = crime_gdf['END_DATE'].dt.hour
crime_gdf['shift'] = crime_gdf['hour'].apply(assign_shift_int)

In [34]:
# Functions have built in print statements
# Collect user data for A to B travel
geo_locations, text_locations = loc_prompt()
geo_center, geo_dist, start_lat, start_lon, end_lat, end_lon = center(geo_locations)
departure_time = dep_time()
departure_time_shift = assign_shift_string(departure_time)

# filter for crime associated with shift of interest
crime_gdf = crime_gdf[crime_gdf['shift'] == departure_time_shift]

# build required part of graph
graph = ox.graph_from_point(geo_center, dist=geo_dist, network_type='walk')

Location not found. Please query again.
Location not found. Please query again.
Location not found. Please query again.
Location not found. Please query again.
Location not found. Please query again.
Start Point: (38.8901225, -77.0495523) '2 Lincoln Memorial Cir NW, Washington, DC 20002'
End Point: (38.9040321, -77.039735) '1101 Connecticut Ave NW, Washington, DC 20036'


In [35]:
# find nearest node to be origin/destination
origin_node = ox.distance.nearest_nodes(graph, start_lon, start_lat)
destination_node = ox.distance.nearest_nodes(graph, end_lon, end_lat)

Crime density weight


In [36]:
graph = ox.project_graph(graph)  # projects to UTM automatically
nodes, edges = ox.graph_to_gdfs(graph, nodes=True, edges=True)

crime_gdf = crime_gdf.to_crs(edges.crs)

In [37]:
# calculate baseline the shortest route from A to B
route_short = nx.shortest_path(graph, origin_node, destination_node, weight = 'length', method = 'dijkstra')

In [38]:
buffer_dist = 41.5  # this is the average edge length, can be adjusted for different routing characteristics

edges['geometry_buffer'] = edges.geometry.buffer(buffer_dist)
edges_buffered = edges.set_geometry('geometry_buffer')

# spatial join: crimes within each edge buffer
joined = gpd.sjoin(crime_gdf, edges_buffered, how='left', predicate='within')

In [39]:
crime_counts = joined.groupby(['u', 'v', 'key']).size()

edges['crime_count'] = edges.index.map(crime_counts).fillna(0)
edges['crime_density'] = edges['crime_count'] / edges['length']

In [40]:
alpha = 100  # HYPERPARAM: tune this (0 -> 100+) # TODO: Incorporate into function to allow user to dial in?

In [41]:
# calculate the 'crime_weight' that will be used as the weight to calculate optimate route while avoiding high crime density edges

cd_max = edges['crime_density'].max()
if cd_max == 0:
    print('WARNING: fallback to \'length\' occurred')
    edges['crime_weight'] = edges['length']
else:
    edges['crime_density_norm'] = (edges['crime_density'] / cd_max)
    edges['crime_weight'] = edges['length'] * (1 + (alpha * edges['crime_density_norm']))

In [42]:
graph = ox.graph_from_gdfs(nodes, edges) # rebuilding with updated edges values
# create the shortest route with crime weight
route_crime = nx.shortest_path(graph, origin_node, destination_node, weight = 'crime_weight', method = 'dijkstra')

In [43]:
# Claude solution for mis-matched graphical representations
graph_latlon = ox.project_graph(graph, to_crs='epsg:4326')

route_coords_short = [
    (graph_latlon.nodes[node]['y'], graph_latlon.nodes[node]['x'])
    for node in route_short
]

route_coords_crime = [
    (graph_latlon.nodes[node]['y'], graph_latlon.nodes[node]['x'])
    for node in route_crime
]

origin_xy = (graph_latlon.nodes[origin_node]['y'], graph_latlon.nodes[origin_node]['x'])
destination_xy = (graph_latlon.nodes[destination_node]['y'], graph_latlon.nodes[destination_node]['x'])

Folium Graph


In [44]:
from folium.plugins import ScrollZoomToggler

m = folium.Map(
    location = geo_center,
    zoom_start = 12 #TODO: zoom should start dependant on how large the route is
)

ScrollZoomToggler().add_to(m)
folium.PolyLine(route_coords_short, color = 'red', weight = 7, opacity = 0.5).add_to(m)
folium.PolyLine(route_coords_crime, color = 'blue', weight = 3, opacity = 0.5).add_to(m)

folium.Marker(
    location = origin_xy,
    icon = folium.Icon(color='green', icon = 'flag', prefix='fa')
).add_to(m)
folium.Marker(
    location = destination_xy,
    icon = folium.Icon(color='red', icon = 'flag', prefix = 'fa')
).add_to(m)

m

In [45]:
compute_route_metrics(graph, origin_node, destination_node, departure_time, text_locations[0], text_locations[1])


ROUTE COMPARISON
From '2 Lincoln Memorial Cir NW, Washington, DC 20002' to '1101 Connecticut Ave NW, Washington, DC 20036' at 17:45 EST

   [Shortest Distance Route]                    [Safest (Crime-Weighted) Route]
  Total Distance (meters): 2,230.78          Total Distance (meters):        2,261.75
  Crime-Weighted Distance: 2,395.36          Crime-Weighted Distance:        2,291.67

DIFFERENCE
  Extra Distance for Safety (m):  30.97
  Decrease in crime exposure at 41.5 meter render distance: 81.57%
